# Goal
Compute proxy visual features that roughly correspond to dermatological concepts like:
- Asymmetry
- Border irregularity
- Color variation
- Diameter / size


| Concept             | Automatic proxy                   | What it reveals    |
| ------------------- | --------------------------------- | ------------------ |
| Asymmetry           | Difference between flipped halves | Shape imbalance    |
| Border irregularity | Boundary curvature variance       | Jagged edges       |
| Color variation     | Delta-E (color deviation)         | Pigment unevenness |
| Diameter            | Radial distance map               | Lesion size cue    |



## A. Asymmetry map

Idea: Malignant lesions are often asymmetric.

How:
- Compute lesion mask centroid.
- Rotate and flip the lesion across its vertical/horizontal axes.
- Subtract mirrored halves → the absolute difference gives an asymmetry heatmap (highlighting irregular sides).

In [ ]:
import cv2, numpy as np
flipped = cv2.flip(mask, 1)
diff = cv2.absdiff(mask, flipped)
asymmetry_map = diff / diff.max()

## B. Border irregularity map

Idea: Jagged, uneven borders often indicate malignancy.

How:

- Find the contour from the mask.
- Compute curvature or local variance of distance from centroid to boundary points.
- Mark boundary points with high curvature or high variance as “irregular”.
- Spread that information slightly inward (dilate 3–5 pixels) to create a border irregularity map.

Result: A thin rim around lesion showing irregular edges.

## C. Color variation map

Idea: Malignant lesions have uneven color distributions.
How:

Convert lesion region to LAB or HSV color space.

Compute per-pixel color difference from lesion mean (ΔE in LAB).

Normalize → pixels with higher ΔE = higher color variance.

In [ ]:
deltaE = np.sqrt((L - L.mean())**2 + (a - a.mean())**2 + (b - b.mean())**2)
color_var_map = deltaE / deltaE.max()


## D. Diameter map (size cue)

Idea: Larger lesions (>6 mm) are riskier.

How:

- Use mask area and pixel scale (if known) → scalar feature (not spatial map).

- Or make a radial map from lesion center: each pixel value = distance from centroid normalized by lesion radius.

Method still working with public, automatic features. No clinicians required.

Answering question: Do XAI methods attend to known risk factors (ABCD)?

The evaluation is transformed from pure geometry (mask overlap) to semi-semantic interpretability:  a richer, more original practicum.